# ਪਾਠ 17 - Foundry Local ਅਤੇ Qwen ਨਾਲ ਸਥਾਨਕ AI ਏਜੰਟ ਬਣਾਉਣਾ

ਇਸ ਨੋਟਬੁੱਕ ਵਿੱਚ ਤੁਸੀਂ ਇੱਕ **ਸਥਾਨਕ ਇੰਜੀਨੀਅਰਿੰਗ ਸਹਾਇਕ** ਬਣਾਉਂਦੇ ਹੋ ਜੋ ਪੂਰੀ ਤਰ੍ਹਾਂ ਤੁਹਾਡੇ ਵਰਕਸਟੇਸ਼ਨ 'ਤੇ ਚੱਲਦਾ ਹੈ। ਕਿਸੇ ਵੀ ਸਮੇਂ ਕਲਾਉਡ ਇੰਫਰੰਸ ਦੀ ਵਰਤੋਂ ਨਹੀਂ ਕੀਤੀ ਜਾਂਦੀ। ਸਹਾਇਕ ਇਹ ਕਰ ਸਕਦਾ ਹੈ:

1. **ਟੂਲਜ਼ ਨੂੰ ਕਾਲ ਕਰੋ** Foundry Local ਰਾਹੀਂ Qwen ਫੰਕਸ਼ਨ ਕਾਲਿੰਗ ਦ੍ਵਾਰਾ।
2. **ਸੈਂਡਬਾਕਸਡ ਪ੍ਰੋਜੈਕਟ ਡਾਇਰੈਕਟਰੀ** ਦੇ ਅੰਦਰ ਫਾਇਲਾਂ ਦੀ ਸੂਚੀ ਬਣਾਓ ਅਤੇ ਪੜ੍ਹੋ।
3. **ਕੋਡ ਦਾ ਵਿਸ਼ਲੇਸ਼ਣ ਕਰੋ** ਸਧਾਰਨ ਮੈਟਰਿਕਸ ਨਾਲ।
4. **ਦਸਤਾਵੇਜ਼ੀकरण ਦੀ ਖੋਜ ਕਰੋ** ਸਥਾਨਕ RAG (Chroma) ਨਾਲ।
5. **ਸਥਾਨਕ MCP ਸਰਵਰ ਦੀ ਵਰਤੋਂ ਕਰੋ** (ਜੇ ਕੋਈ ਨਿਰਧਾਰਿਤ ਨਹੀਂ ਹੈ ਤਾਂ ਸਹਿਜਭਾਵ ਨਾਲ ਛੱਡ ਦਿੱਤਾ ਜਾਂਦਾ ਹੈ)।

ਏਜੰਟ ਕੋਡ ਕਲਾਉਡ ਪਾਠਾਂ ਦੇ ਬਿਲਕੁਲ ਸਮਾਨ ਦਿਸਦਾ ਹੈ — ਸਿਰਫ ਕਲਾਇੰਟ ਐਂਡਪੁਆਇੰਟ ਕਲਾਉਡ ਤੋਂ `localhost` ਵੱਲ ਵੱਧਦਾ ਹੈ।


## ਸੈਟਅਪ

ਇਸ ਨੋਟਬੁੱਕ ਨੂੰ ਚਲਾਉਣ ਤੋਂ ਪਹਿਲਾਂ:

1. **Microsoft Foundry Local ਇੰਸਟਾਲ ਕਰੋ** (ਆਪਣੇ OS ਲਈ [ਦਸਤਾਵੇਜ਼](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) ਵੇਖੋ).
2. **Qwen ਮਾਡਲ ਡਾਊਨਲੋਡ ਅਤੇ ਸ਼ੁਰੂ ਕਰੋ:**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. ਹੇਠ ਲਿਖੀਆਂ ਪਾਇਥਨ ਪੈਕੇਜਾਂ ਇੰਸਟਾਲ ਕਰੋ।

ਸਭ ਕੁਝ ਸਥਾਨਕ ਤੌਰ 'ਤੇ ਚਲਦਾ ਹੈ। ਤਕਰੀਬਨ 8 GB RAM ਵਾਲੀ ਮਸ਼ੀਨ ਘੱਟੋ-ਘੱਟ ਯਥਾਰਥਪੂਰਨ ਹੈ।


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## 1. ਫਾਊਂਡਰੀ ਲੋਕਲ ਨਾਲ ਜੁੜੋ

`FoundryLocalManager` ਜੇ ਲੋੜ ਹੋਵੇ ਤਾਂ ਮਾਡਲ ਡਾਊਨਲੋਡ ਕਰਦਾ ਹੈ, ਲੋਕਲ ਸੇਵਾ ਸ਼ੁਰੂ ਕਰਦਾ ਹੈ, ਅਤੇ ਸਾਨੂੰ ਇੱਕ **OpenAI-ਕੰਪੈਟਿਬਲ ਐਂਡਪੋਇੰਟ** ਦਿੰਦਾ ਹੈ। ਫਿਰ ਅਸੀਂ ਸਧਾਰਣ OpenAI SDK ਨੂੰ ਇਸਦੇ ਵੱਲ ਇਸ਼ਾਰਾ ਕਰਦੇ ਹਾਂ। API ਕੁੰਜੀ ਇੱਕ ਲੋਕਲ ਪਲੇਸਹੋਲਡਰ ਹੈ — ਇਸ ਵਿੱਚ ਕੋਈ ਕਲਾਉਡ ਪ੍ਰਮਾਣਪੱਤਰ ਸ਼ਾਮਲ ਨਹੀਂ ਹੈ।


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## 2. ਸਥਾਨਕ ਟੂਲ (ਸੈਂਡਬਾਕਸ ਕੀਤੇ ਫਾਈਲ ਓਪਰੇਸ਼ਨ)

ਅਸੀਂ ਡਿਸ્ક 'ਤੇ ਇੱਕ ਛੋਟਾ ਨਮੂਨਾ ਪ੍ਰੋਜੈਕਟ ਬਣਾਉਂਦੇ ਹਾਂ, ਫਿਰ ਉਸ ਪ੍ਰੋਜੈਕਟ ਰੂਟ ਨਾਲ ਸੀਮਿਤ ਟੂਲ ਪਰਿਭਾਸ਼ਿਤ ਕਰਦੇ ਹਾਂ। ਸੈਂਡਬਾਕਸ ਚੈੱਕ ਸਥਾਨਕ ਤੌਰ 'ਤੇ ਵੀ ਮਹੱਤਵਪੂਰਨ ਹੁੰਦਾ ਹੈ: ਇੱਕ ਟੂਲ ਜੋ ਇੱਛਾ ਅਨੁਸਾਰ ਪਾਥਾਂ ਨੂੰ ਪੜ੍ਹਦਾ ਹੈ, ਤੁਹਾਡੇ ਯੂਜ਼ਰ ਦੀਆਂ ਪਰਮੀਸ਼ਨਾਂ ਨਾਲ ਚੱਲਦਾ ਹੈ ਅਤੇ ਉਹ ਕੁਝ ਵੀ ਛੋਹ ਸਕਦਾ ਹੈ ਜੋ ਤੁਸੀਂ ਛੂਹ ਸਕਦੇ ਹੋ। 


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. ਚਰੋਮਾ ਨਾਲ ਸਥਾਨਕ RAG

ਅਸੀਂ ਇੱਕ ਛੋਟੀ ਦਸਤਾਵੇਜ਼ੀ ਟੁਕੜਿਆਂ ਦੀ ਸੈੱਟ ਨੂੰ ਸਥਾਨਕ ਚਰੋਮਾ ਕਲੈਕਸ਼ਨ ਵਿੱਚ ਏਮਬੈੱਡ ਕਰਦੇ ਹਾਂ। ਚਰੋਮਾ ਪ੍ਰਕਿਰਿਆ ਵਿੱਚ ਚਲਦਾ ਹੈ ਅਤੇ ਡਿਸਕ 'ਤੇ ਵੈਕਟਰਾਂ ਨੂੰ ਸਟੋਰ ਕਰਦਾ ਹੈ — ਕੋਈ ਸਰਵਰ, ਕੋਈ ਕਲਾਉਡ ਨਹੀਂ। `search_docs` ਟੂਲ ਕਵੇਰੀ ਲਈ ਸਭ ਤੋਂ ਸਬੰਧਿਤ ਟੁਕੜਿਆਂ ਨੂੰ ਲੱਭਦਾ ਹੈ।


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## 4. ਟੂਲ-ਕਾਲਿੰਗ ਲੂਪ  

ਹੁਣ ਅਸੀਂ ਟੂਲ ਨੂੰ ਮਾਡਲ ਨਾਲ OpenAI ਟੂਲ ਸਕੀਮਾ ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਰਜਿਸਟਰ ਕਰਦੇ ਹਾਂ ਅਤੇ ਸਟੈਂਡਰਡ ਟੂਲ-ਕਾਲਿੰਗ ਲੂਪ ਚਲਾਉਂਦੇ ਹਾਂ — ਮਾਡਲ ਇੱਕ ਟੂਲ ਦੀ ਮੰਗ ਕਰਦਾ ਹੈ, ਅਸੀਂ ਇਸ ਨੂੰ ਲੋਕਲ ਤੌਰ 'ਤੇ ਚਲਾਉਂਦੇ ਹਾਂ, ਨਤੀਜਾ ਵਾਪਸ ਪਹੁੰਚਾਉਂਦੇ ਹਾਂ, ਅਤੇ ਦੁਹਰਾਉਂਦੇ ਹਾਂ ਜਦ ਤੱਕ ਮਾਡਲ ਆਖਰੀ ਜਵਾਬ ਨਹੀਂ ਦਿੰਦਾ। Qwen ਦਾ ਭਰੋਸੇਯੋਗ ਫੰਕਸ਼ਨ ਕਾਲਿੰਗ ਹੀ ਇਸ ਕੰਮ ਨੂੰ ਡਿਵਾਈਸ 'ਤੇ ਸੰਭਵ ਬਣਾਉਂਦਾ ਹੈ।  


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## 5. ਲੋਕਲ MCP (ਵਿਕਲਪੀ)

MCP ਇੱਕ ਟ੍ਰਾਂਸਪੋਰਟ ਹੈ, ਕੋਈ ਕਲਾਊਡ ਸਰਵਿਸ ਨਹੀਂ — ਇੱਕ MCP ਸਰਵਰ ਨੂੰ `stdio` 'ਤੇ ਲੋਕਲ ਪ੍ਰਕਿਰਿਆ ਵਜੋਂ ਚਲਾਇਆ ਜਾ ਸਕਦਾ ਹੈ। ਹੇਠਾਂ ਦਿੱਤਾ ਗਿਆ ਸੈੱਲ ਦਿਖਾਉਂਦਾ ਹੈ ਕਿ ਤੁਸੀਂ ਜੇਕਰ ਕੋਈ ਲੋਕਲ MCP ਸਰਵਰ (ਉਦਾਹਰਨ ਵਜੋਂ ਫਾਇਲਸਿਸਟਮ ਸਰਵਰ) ਕਾਨਫਿਗਰ ਕੀਤਾ ਹੋਵੇ ਤਾਂ ਉਸ ਨਾਲ ਕਿਵੇਂ ਕਨੇਕਟ ਕਰੋਗੇ। ਜਦ `LOCAL_MCP_COMMAND` ਸੈੱਟ ਨਹੀਂ ਹੁੰਦਾ, ਤਾਂ ਇਹ ਸੁਚਾਰੂ ਤਰੀਕੇ ਨਾਲ ਛੱਡ ਦਿੰਦਾ ਹੈ, ਤਾਂ ਜੋ ਨੋਟਬੁੱਕ ਅਜੇ ਵੀ ਆਖਰੀ ਤੱਕ ਚੱਲਦਾ ਰਹੇ।

ਸੁਰੱਖਿਆ ਟਿਪ्पਣੀ: ਇੱਕ ਲੋਕਲ MCP ਸਰਵਰ ਤੁਹਾਡੇ ਯੂਜ਼ਰ ਦੀਆਂ ਅਧਿਕਾਰਾਂ ਨਾਲ ਚਲਦਾ ਹੈ। ਇਸਨੂੰ ਕਿਸੇ ਪ੍ਰੋਜੈਕਟ ਡਾਇਰੈਕਟਰੀ ਤੱਕ ਸੀਮਤ ਕਰੋ ਅਤੇ ਇਸਦੇ ਨਤੀਜੇ ਕਾਰਵਾਈ ਕਰਨ ਤੋਂ ਪਹਿਲਾਂ ਸਹੀ ਤੌਰ 'ਤੇ ਜਾਂਚੋ।


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## ਸਾਰ

ਤੁਸੀਂ ਇੱਕ ਇੰਜੀਨੀਅਰਿੰਗ ਸਹਾਇਕ ਬਣਾਇਆ ਜਿਹੜਾ ਪੂਰੀ ਤਰ੍ਹਾਂ ਤੁਹਾਡੇ ਮਸ਼ੀਨ 'ਤੇ ਚਲਦਾ ਹੈ:

- **Foundry Local** ਨੇ ਇੱਕ **Qwen** ਮਾਡਲ ਨੂੰ OpenAI-ਸਮਰਥਿਤ ਐਂਡਪੋਇੰਟ ਦੇ ਥੱਲੇ ਪਰੋਵਾਈਡ ਕੀਤਾ — ਤਾਂ ਜੋ ਏਜੰਟ ਕੋਡ ਕਲਾਉਡ ਸਬਕਾਂ ਨਾਲ ਮੇਲ ਖਾਂਦਾ ਰਹੇ।
- **Sandboxed tools** ਨੇ ਏਜੰਟ ਨੂੰ ਪ੍ਰੋਜੈਕਟ ਡਾਇਰੈਕਟਰੀ ਛੱਡੇ ਬਿਨਾਂ ਫਾਇਲ ਐਕਸੈਸ ਅਤੇ ਕੋਡ ਵਿਸ਼ਲੇਸ਼ਣ ਦਿੱਤੀ।
- **Chroma** ਨੇ ਦਸਤਾਵੇਜ਼ੀਕਰਨ ਉੱਤੇ **ਲੋਕਲ RAG** ਪ੍ਰਦਾਨ ਕੀਤਾ।
- **Local MCP** ਨੇ ਦਿਖਾਇਆ ਕਿ MCP ਪਰਿਬੇਸ਼ ਨੂੰ ਆਫਲਾਈਨ ਕਿਵੇਂ ਦੁਬਾਰਾ ਵਰਤਿਆ ਜਾ ਸਕਦਾ ਹੈ।

ਕਿਸੇ ਵੀ ਸਮੇਂ ਕਲਾਉਡ ਇੰਫਰੈਂਸ ਦੀ ਵਰਤੋਂ ਨਹੀਂ ਕੀਤੀ ਗਈ ਸੀ।

### ਚੁਣੌਤੀ

ਲੋਕਲ ਏਜੰਟ ਨੂੰ ਵਧਾਓ ਤਾਂ ਕਿ:

1. **ਕਈ MCP ਸਰਵਰਾਂ ਨਾਲ ਕੰਮ ਕਰ ਸਕੇ** — ਫਾਇਲਸਿਸਟਮ ਸਰਵਰ ਅਤੇ ਗਿਟ ਸਰਵਰ ਨੂੰ ਕਨੈਕਟ ਕਰੋ ਅਤੇ ਏਜੰਟ ਨੂੰ ਉਹਨਾਂ ਵਿੱਚੋਂ ਚੁਣਨ ਦਿਓ।
2. **ਲੋਕਲ ਮੇਮੋਰੀ ਦੀ ਵਰਤੋਂ ਕਰੋ** — ਇੱਕ ਛੋਟੀ ਗੱਲਬਾਤ ਦਾ ਇਤਿਹਾਸ ਡਿਸਕ 'ਤੇ ਸਟੋਰ ਕਰੋ ਤਾਂ ਜੋ ਸਹਾਇਕ ਨੋਟਬੁੱਕ ਰੀਸਟਾਰਟਾਂ ਦੇ ਦੌਰਾਨ ਪਹਿਲੀਆਂ ਗੱਲਾਂ ਯਾਦ ਰਖੇ।
3. **ਲੋਕਲ ਮਲਟੀ-ਏਜੰਟ ਸੰਚਾਲਨ (orchestration) ਦਾ ਸਮਰਥਨ ਕਰੋ** — ਇੱਕ ਦੂਸਰਾ ਲੋਕਲ ਏਜੰਟ (ਜਿਵੇਂ ਸਮੀਖਿਆਕਾਰ) ਸ਼ਾਮਲ ਕਰੋ ਅਤੇ ਦੋਹਾਂ ਨੂੰ ਕਿਸੇ ਕੰਮ ਉੱਤੇ ਸਹਿਯੋਗ ਕਰਵਾਓ।

ਅਗਲੇ ਸਬਕ ਵਿੱਚ ਤੁਸੀਂ ਸਿੱਖੋਗੇ ਕਿ ਡਿਪਲੋਅਡ ਕੀਆ ਗਿਆ AI ਏਜੰਟ ਕਿਵੇਂ ਸੁਰੱਖਿਅਤ ਕੀਤਾ ਜਾਏ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸਵੀਕਾਰੋਪਣ**:
ਇਸ ਦਸਤਾਵੇਜ਼ ਦਾ ਅਨੁਵਾਦ ਏਆਈ ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸਹੀਤਾਵਾਂ ਲਈ ਯਤਨਸ਼ੀਲ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਰੱਖੋ ਕਿ ਸਵੈਚਾਲਿਤ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਸਮੱਤਿਆਵਾਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਅਧਿਕਾਰਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਜਰੂਰੀ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫ਼ਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੇ ਉਪਯੋਗ ਤੋਂ ਪੈਦਾ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਿਮੀਆਂ ਜਾਂ ਗਲਤ ਵਿਆਖਿਆਵਾਂ ਲਈ ਜਵਾਬਦੇਹ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
